# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Bước 1: thống kê split, kiểm tra không phiên nào trùng, vẽ 6 ảnh có box GT
from scripts.labcommon import read_jsonl
sp = {s: read_jsonl(f'data/splits/{s}.jsonl') for s in ('train', 'val', 'test')}
ses = {s: {e['session'] for e in v} for s, v in sp.items()}
assert not (ses['train'] & ses['val'] or ses['train'] & ses['test'] or ses['val'] & ses['test'])
!python -m vlmlab.viz --split train --gt-only --out report/figures/train_gt --n 6

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Bước 2: vlmlab/labels.py (yolo_to_box, objects_to_answer, prompt) + test
from vlmlab.labels import objects_to_answer, make_prompt
print(make_prompt()); print(objects_to_answer(sp['train'][0]['objects']))
!python -m pytest -q vlmlab/tests

## 3. E0: zero-shot

In [ ]:
# Bước 3: E0 zero-shot (vlmlab/run.py: load_model, cmd_infer)
!python -m vlmlab.run inspect   # tên module, số tham số, 220 token ảnh, mask loss
!python -m vlmlab.run infer --split test --out work/e0.jsonl

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Bước 4: vlmlab/run.py: encode_example() (mask prompt + token ảnh = -100) và collate()
# kiểm tra đã chạy trong `inspect` ở trên: report/logs/inspect.txt

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Bước 5: vlmlab/run.py: apply_setup() — LoRA regex LLM (252) / LLM+ViT (348), projector fp32

## 6. Vòng huấn luyện

In [ ]:
# Bước 6: vòng train fp16 + GradScaler, lưu adapter + projector.pt + run.json
for s in ('e1', 'e2', 'e3'):
    !python -m vlmlab.run train --setup {s} --out work/{s}

## 7. Suy luận và chấm E1–E3

In [ ]:
# Bước 7: suy luận + chấm (meta lấy từ work/eN/run.json)
import json
for s in ('e1', 'e2', 'e3'):
    !python -m vlmlab.run infer --split test --adapter work/{s} --out work/{s}.jsonl
    r = json.load(open(f'work/{s}/run.json'))
    !python scripts/score.py --pred work/{s}.jsonl --name {s.upper()} --out report/evidence/{s}.json --meta trainable_params={r['trainable_params']} --meta peak_vram_gb={r['peak_vram_gb']} --meta train_minutes={r['train_minutes']}

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# Bước 8 (E4) và 9 (E5): xem REPORT.md để biết biến đã chọn và lệnh cụ thể
# E5: !python -m vlmlab.grpo --init work/<best> --out work/e5